# Part 3 — Predictive modelling: racialised rate as a function of socioeconomic/political structure

**Scope for this pass** (deliberately simple, per the plan set out in `02_socioeconomic_political_relations.ipynb` §2.8): a single multivariate model with `pool_count` included as a covariate, rather than the one-at-a-time partial correlations §2.4b had to rely on. This directly follows §2.8's RQ2 spec: *"a negative-binomial GLM with a review-volume offset, `pool_count`/venue-density included as a covariate rather than partialled out separately."* This is a preliminary pass on the working assumption that the IAA and robustness work already completed is sufficient to trust the `racialised` label going into a model — both the label and the model itself are expected to be iterated on further.

## 3.0 Setup — continuing from Parts 1 and 2

Rebuilds `analysis_df` exactly as in `02_socioeconomic_political_relations.ipynb` (§2.1 merge, §2.3 derived percentage columns, same `PREDICTOR_COLS`), rather than depending on any in-memory state from that notebook.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
OUTPUTS_DIR = ANALYSIS_DIR / "notebook_outputs"
FIGURES_DIR = ANALYSIS_DIR / "figures"
PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"

political = pd.read_csv(SOCIO_DIR / "federal_baseline_eng_selected_political.csv")
const_agg = pd.read_parquet(OUTPUTS_DIR / "const_agg.parquet")

assert const_agg.shape[0] == 299, "expected 299 constituencies"

political_extra = political.drop(columns=["state", "constituency"])
analysis_df = const_agg.merge(political_extra, on="constituency_code", how="left")
assert analysis_df["total_population"].notna().all(), "every constituency should have matched socioeconomic data"

pop = analysis_df["total_population"]
analysis_df["foreign_population_pct"] = analysis_df["foreign_population"] / pop * 100
for age_col in ["age_under_18_population", "age_18_24_population", "age_25_34_population",
                "age_35_59_population", "age_60_74_population", "age_75_plus_population"]:
    analysis_df[age_col.replace("_population", "_pct")] = analysis_df[age_col] / pop * 100
analysis_df["disposable_income_eur_per_capita"] = analysis_df["disposable_income_total_eur"] / pop
analysis_df["gdp_eur_per_capita"] = analysis_df["gdp_total_eur"] / pop
analysis_df["employees_pct_of_pop"] = analysis_df["employees_total"] / pop * 100
analysis_df["sgb_ii_recipients_pct"] = analysis_df["sgb_ii_recipients_total"] / pop * 100

FULL_PREDICTOR_COLS = [
    "foreign_population_pct", "age_under_18_pct", "age_18_24_pct", "age_25_34_pct",
    "age_60_74_pct", "age_75_plus_pct", "disposable_income_eur_per_capita", "gdp_eur_per_capita",
    "employees_pct_of_pop", "sgb_ii_recipients_pct", "unemployment_total", "turnout_pct",
    "second_vote_spd_pct", "second_vote_cdu_csu_pct", "second_vote_greens_pct",
    "second_vote_afd_pct", "second_vote_left_pct", "second_vote_bsw_pct",
]

analysis_df["log_social_reviews"] = np.log(analysis_df["social_reviews"])
analysis_df["log_total_text_reviews"] = np.log(analysis_df["total_text_reviews"])
analysis_df["racialised_rate_of_all"] = analysis_df["racialised_reviews"] / analysis_df["total_text_reviews"]

print(f"analysis_df: {analysis_df.shape[0]} constituencies x {analysis_df.shape[1]} cols")
print(f"{len(FULL_PREDICTOR_COLS)} \u00a72.4 predictor candidates + pool_count")
analysis_df[["constituency", "state", "racialised_reviews", "social_reviews", "pool_count", "reliable_n"]].head()

## 3.1 Model specification

Two choices, each following directly from what Parts 1-2 already established:

- **Outcome is the count, not the rate, with an offset.** `racialised_reviews` (not `racialised_rate`) is the response, with `log(social_reviews)` as an offset — the standard way to model a rate that comes from a variable-sized denominator (analogous to person-time in epidemiology). It lets the model use the count's actual sampling variance instead of treating `racialised_rate` as a continuous measurement with constant precision, which it isn't: a constituency with 300 social reviews and one with 3,000 have very different sampling noise around the same rate.
- **Negative binomial, not Poisson.** §2.6 already found genuine spatial clustering (Moran's I = 0.28, p = 0.001) — clustering implies extra between-constituency variance beyond what a Poisson model's mean-equals-variance assumption allows. We fit the Poisson model first specifically to demonstrate this overdispersion numerically before moving to NB2, rather than asserting it.

What still needs deciding is *which* covariates go in alongside `pool_count` — §3.2 below.

In [ ]:
X_full = sm.add_constant(analysis_df[["pool_count"] + FULL_PREDICTOR_COLS])
y = analysis_df["racialised_reviews"]
offset = analysis_df["log_social_reviews"]

poisson_full = sm.GLM(y, X_full, family=sm.families.Poisson(), offset=offset).fit()
dispersion_full = poisson_full.pearson_chi2 / poisson_full.df_resid
print(f"Poisson GLM (pool_count + all {len(FULL_PREDICTOR_COLS)} predictors): "
      f"Pearson chi2 / df = {dispersion_full:.2f} (>>1 => overdispersed, NB2 needed either way)")

Pearson chi2/df = 7.99, well above the 1.0 a well-specified Poisson model would show — confirms overdispersion, consistent with §2.6's spatial clustering finding, and is the empirical justification for using NB2 rather than Poisson throughout the rest of this notebook (the reduced-set Poisson model in §3.3 shows the same pattern, chi2/df = 9.46).

## 3.2 Predictor set: why not just put all 18 in?

§2.4b/§2.8 called for "`pool_count` as a covariate, not partialled out separately" precisely *because* the §2.4 predictors are themselves heavily intercorrelated with `pool_count` and with each other (deprivation, age structure, and vote share all move together). The natural first attempt is therefore the maximal one: `pool_count` + all 18 predictors in a single NB2. We check that model's viability with VIF (variance inflation factor) *before* trusting any coefficient from it — a VIF above ~10 means a predictor's estimated effect is essentially undetermined by the data, drowned out by its correlation with the other covariates in the model, not evidence of a null effect.

In [ ]:
vif_full = pd.DataFrame({
    "predictor": X_full.columns,
    "vif": [variance_inflation_factor(X_full.values, i) for i in range(X_full.shape[1])],
}).drop(index=0)  # drop the constant term's VIF, which is uninformative here
vif_full.sort_values("vif", ascending=False)

Confirms the concern directly: VIF exceeds 10 for **11 of the 19 covariates**, topping out at **79.7 for `second_vote_afd_pct`**, 58.6 for `second_vote_cdu_csu_pct`, and 38.4 for `second_vote_greens_pct` — the vote-share variables are so intercorrelated with each other (they're shares of the same total, on top of §2.4b's already-established correlation with `pool_count`) that no single one of them has an independently estimable effect in a model that includes all of them at once. `pool_count` itself has a comfortable VIF of 2.3 — it's the combination of many mutually collinear vote-share/deprivation variables that's unusable, not `pool_count`'s inclusion.

Given that, we don't fit the 19-covariate NB2 as the primary model at all — attempting it directly confirms the concern: the MLE fails to converge (Hessian inversion fails, so no standard errors are even computable) and 5-fold cross-validation on it produces predictions that are numerically unstable across folds. That's not a coding bug, it's the expected consequence of asking an MLE to separate the effects of 19 covariates, several pairwise/multiply collinear, from n=299 constituencies.

Instead we use the **6-predictor set §2.4b already validated as robust to the `pool_count` confound** — `sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`, `turnout_pct`, `second_vote_spd_pct`, `second_vote_bsw_pct` (the only ones that stayed FDR-significant once `pool_count` was partialled out one at a time). Moving from "one-at-a-time partial correlation" to "one joint multivariate model" only for *this* set is the actual intuitive next step from §2.4b/§2.8 — it keeps the covariate-to-observation ratio sane (7 covariates, n=299) while still jointly modelling the interactions those 6 predictors have with each other and with `pool_count`, which one-at-a-time partial correlation could not do.

In [ ]:
REDUCED_PREDICTOR_COLS = [
    "sgb_ii_recipients_pct", "foreign_population_pct", "unemployment_total",
    "turnout_pct", "second_vote_spd_pct", "second_vote_bsw_pct",
]
MODEL_COLS = ["pool_count"] + REDUCED_PREDICTOR_COLS

X = sm.add_constant(analysis_df[MODEL_COLS])

vif_reduced = pd.DataFrame({
    "predictor": X.columns,
    "vif": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
}).drop(index=0)
vif_reduced.sort_values("vif", ascending=False)

Much better, but not perfect: `sgb_ii_recipients_pct` (VIF 26.8) and `unemployment_total` (VIF 25.8) are still highly collinear with each other — unsurprising, since both are deprivation measures — while `pool_count` and the remaining four predictors are all comfortably below 4. This is a known limitation of this pass (flagged again in §3.7): the model below can jointly estimate "the deprivation cluster's" effect reasonably well, but can't cleanly separate `sgb_ii_recipients_pct`'s individual contribution from `unemployment_total`'s.

## 3.3 Fitting the negative binomial model on the reduced set

In [ ]:
poisson_model = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset).fit()
dispersion = poisson_model.pearson_chi2 / poisson_model.df_resid
print(f"Poisson GLM (reduced set): Pearson chi2 / df = {dispersion:.2f}")

nb_full = NegativeBinomial(y, X, offset=offset).fit(disp=False)
print(nb_full.summary())

In [ ]:
irr_table = pd.DataFrame({
    "coef": nb_full.params,
    "irr": np.exp(nb_full.params),
    "irr_ci_lo": np.exp(nb_full.conf_int()[0]),
    "irr_ci_hi": np.exp(nb_full.conf_int()[1]),
    "p": nb_full.pvalues,
}).drop(index=["alpha", "const"], errors="ignore")
irr_table.sort_values("p")

**Takeaway.** The model converges cleanly this time (`converged: True`), and is overall highly significant (LLR p = 6.8e-30 vs. the null offset-only model), with a modest pseudo-R² of 0.066 — expected for a preliminary ecological model of a rare, noisy outcome. Two covariates are clearly significant once everything is modelled jointly:

- **`foreign_population_pct`** (IRR = 1.034, 95% CI [1.019, 1.049], p < 0.0001): each 1 percentage-point increase in foreign-population share is associated with a 3.4% higher expected racialised-review rate, holding `pool_count` and the other five predictors constant. This is the one predictor that has now survived three separate checks — §2.4's univariate correlation (ρ=0.46), §2.4b's `pool_count`-partialled correlation (ρ=0.28), and this joint multivariate model — making it the most defensible finding across all three notebooks so far.
- **`pool_count`** (IRR = 0.989, 95% CI [0.980, 0.997], p = 0.010): each additional pool in a constituency is associated with a ~1.1% lower expected rate, confirming §2.4b's confound finding survives being modelled as a covariate rather than partialled out.

Everything else is weaker than §2.4b's partial correlations suggested: `sgb_ii_recipients_pct` (p=0.070) and `turnout_pct` (p=0.096) are only marginal, and **`unemployment_total`'s sign flips to negative** (coef=-0.068, p=0.317, not significant) — almost certainly an artifact of its VIF=25.8 collinearity with `sgb_ii_recipients_pct` (§3.2) rather than a real reversal of the §2.4/§2.4b positive association; the two deprivation variables are fighting over the same shared variance once modelled together. `second_vote_spd_pct` and `second_vote_bsw_pct` are both null here (p=0.849, p=0.461) — §2.4b's partial correlations for these two (ρ=0.14, ρ=0.15) don't survive being placed in a joint model alongside the stronger deprivation/composition predictors.

In [ ]:
X_reduced_null = sm.add_constant(analysis_df[["pool_count"]])
nb_pool_only = NegativeBinomial(y, X_reduced_null, offset=offset).fit(disp=False)

lr_stat = 2 * (nb_full.llf - nb_pool_only.llf)
df_diff = len(nb_full.params) - len(nb_pool_only.params)
p_lr = stats.chi2.sf(lr_stat, df_diff)

print(f"pool_count-only NB2: log-likelihood = {nb_pool_only.llf:.1f}")
print(f"pool_count + {len(REDUCED_PREDICTOR_COLS)} predictors NB2: log-likelihood = {nb_full.llf:.1f}")
print(f"likelihood-ratio test: LR = {lr_stat:.1f}, df = {df_diff}, p = {p_lr:.2e}")

**Takeaway.** The 6 predictors jointly add real explanatory power beyond `pool_count` alone: LR = 77.2 on 6 df, p = 1.3e-14. This directly answers §2.8's RQ2 — "is there a real multivariate model here, or does pool_count/venue density explain the pattern by itself" — and confirms that no, deprivation and composition variables (mainly `foreign_population_pct`) explain something venue density alone does not.

## 3.3b Does the same model explain the racial/all-reviews rate?

§2.4's rate-comparison check found the two racialised-rate definitions (`/social` vs. `/all text reviews`) tell an almost identical correlational story. This repeats that check at the regression stage: same count (`racialised_reviews`), same `MODEL_COLS` (VIF is a property of the predictor matrix alone, §3.2, so the reduced set doesn't need re-deriving), only the offset changes -- `log_total_text_reviews` in place of `log_social_reviews` -- to see whether the joint model tells a different story once the denominator is diluted with reviews that were never in scope for racialised-discourse coding.

In [ ]:
offset_rate2 = analysis_df["log_total_text_reviews"]

nb_full_rate2 = NegativeBinomial(y, X, offset=offset_rate2).fit(disp=False)
print(nb_full_rate2.summary())

In [ ]:
irr_table_rate2 = pd.DataFrame({
    "coef": nb_full_rate2.params,
    "irr": np.exp(nb_full_rate2.params),
    "irr_ci_lo": np.exp(nb_full_rate2.conf_int()[0]),
    "irr_ci_hi": np.exp(nb_full_rate2.conf_int()[1]),
    "p": nb_full_rate2.pvalues,
}).drop(index=["alpha", "const"], errors="ignore")

model_comparison = irr_table[["irr", "p"]].join(
    irr_table_rate2[["irr", "p"]], lsuffix="_social", rsuffix="_all"
)
model_comparison.sort_values("p_social")

In [ ]:
nb_pool_only_rate2 = NegativeBinomial(y, X_reduced_null, offset=offset_rate2).fit(disp=False)

lr_stat_rate2 = 2 * (nb_full_rate2.llf - nb_pool_only_rate2.llf)
p_lr_rate2 = stats.chi2.sf(lr_stat_rate2, len(nb_full_rate2.params) - len(nb_pool_only_rate2.params))
print(f"pool_count + {len(REDUCED_PREDICTOR_COLS)} predictors NB2 (racial/all offset): "
      f"log-likelihood = {nb_full_rate2.llf:.1f}")
print(f"likelihood-ratio test vs. pool_count-only: LR = {lr_stat_rate2:.1f}, p = {p_lr_rate2:.2e}")

**Takeaway.** Swapping the offset from `log_social_reviews` to `log_total_text_reviews` barely changes the model: the same two covariates are significant, with the same signs and similar magnitudes -- `foreign_population_pct` (IRR 1.032 vs. 1.034, p<0.0001 both) and `pool_count` (IRR 0.987 vs. 0.989, p=0.004 vs. p=0.010) -- and every other covariate stays non-significant in both fits (`sgb_ii_recipients_pct` p=0.099 vs. 0.070, `turnout_pct` p=0.131 vs. 0.096, `unemployment_total` p=0.370 vs. 0.317, `second_vote_spd_pct`/`second_vote_bsw_pct` both null). The joint LR test against the `pool_count`-only null is highly significant either way (LR=72.2, p=1.4e-13 for `/all`, vs. LR=77.2, p=1.3e-14 for `/social`).

This confirms at the regression stage what §2.4's correlation-level check already found: the two racialised-rate definitions are redundant here, not just correlated. Nothing downstream (§3.4 onward) needs to be re-run for `racialised_rate_of_all` -- Part 1's `/social` choice is the one worth carrying forward, and the rest of this notebook continues to use it as the primary DV.

## 3.4 Does the model actually predict better, out-of-sample?

§3.3 tests whether the predictors are jointly *significant* in-sample; that's a different question from whether they improve *predictive* performance on constituencies the model hasn't seen. We run 5-fold cross-validation comparing the `pool_count`-only model against the reduced model, scoring each fold's held-out predictions on the rate scale (predicted rate vs. actual `racialised_rate`) with mean absolute error — a simple, interpretable predictive check appropriate for a preliminary pass on n=299.

In [ ]:
def cv_mae(cols, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    X_full_cv = sm.add_constant(analysis_df[cols])
    y_full = analysis_df["racialised_reviews"]
    offset_full = analysis_df["log_social_reviews"]
    rate_actual = analysis_df["racialised_rate"].values
    errors = []
    for train_idx, test_idx in kf.split(np.arange(len(analysis_df))):
        model = NegativeBinomial(
            y_full.iloc[train_idx], X_full_cv.iloc[train_idx], offset=offset_full.iloc[train_idx]
        ).fit(disp=False)
        pred_rate = np.exp(X_full_cv.iloc[test_idx].values @ model.params.drop("alpha").values)
        errors.append(np.abs(pred_rate - rate_actual[test_idx]))
    return np.concatenate(errors)

mae_pool_only = cv_mae(["pool_count"])
mae_reduced = cv_mae(MODEL_COLS)

print("5-fold CV mean absolute error (percentage points of racialised_rate):")
print(f"  pool_count-only model: {mae_pool_only.mean() * 100:.3f} pp")
print(f"  reduced model:         {mae_reduced.mean() * 100:.3f} pp")

**Takeaway.** The reduced model beats the `pool_count`-only model out-of-sample too, not just in-sample: 0.610pp mean absolute error vs. 0.691pp — an ~12% relative reduction. The improvement is real but modest, consistent with the in-sample picture above: one strong predictor (`foreign_population_pct`) plus a weak confound-adjustment covariate (`pool_count`), rather than a large multi-predictor gain. That's expected for a first predictive pass on a rare, noisy ecological outcome with n=299, and it sets a concrete baseline (0.61pp MAE) that future iterations of this model can be compared against.

## 3.4b Does `population_density_per_km2` outperform `pool_count` as the density covariate?

`02_socioeconomic_political_relations.ipynb` §2.7-2.8 added `population_density_per_km2` to the data pipeline and found it out-predicts `pool_count` in raw correlation terms (density vs `racialised_rate` ρ=0.54 vs `pool_count` vs `racialised_rate` ρ=-0.49), while the two are themselves strongly collinear (Spearman ρ=-0.80: more pools ↔ lower density, confirming `pool_count` is essentially a noisy proxy for density). That reframes a question this notebook's §3.1-3.3 model didn't ask: is `pool_count` in `MODEL_COLS` actually the best choice of density covariate, or would swapping in `population_density_per_km2` (or including both) fit better?

We compare three specifications, all using the same 6-predictor `REDUCED_PREDICTOR_COLS` set from §3.2:
1. **`pool_count`-based** (the existing §3.3 model, `nb_full`).
2. **Density-only**: `population_density_per_km2` in place of `pool_count`.
3. **Both together**: `pool_count` and `population_density_per_km2` jointly (checking whether their ρ=-0.80 collinearity makes this poorly identified).

First, though: `population_density_per_km2` ranges from ~37/km² in rural constituencies to ~12,190/km² in dense Berlin seats, which is likely far more right-skewed than `pool_count` and may need a log-transform before it can be used as a sane linear covariate in the GLM's linear predictor.

In [ ]:
print("Skewness (right-skew if >> 0):")
print(f"  pool_count:                          {analysis_df['pool_count'].skew():.3f}")
print(f"  population_density_per_km2 (raw):    {analysis_df['population_density_per_km2'].skew():.3f}")
print(f"  log(population_density_per_km2):     {np.log(analysis_df['population_density_per_km2']).skew():.3f}")

analysis_df["log_population_density"] = np.log(analysis_df["population_density_per_km2"])

print(f"\nSpearman rho, pool_count vs population_density_per_km2 (raw): "
      f"{stats.spearmanr(analysis_df['pool_count'], analysis_df['population_density_per_km2'])[0]:.3f}")

`population_density_per_km2`'s skewness (~3.3) is roughly 3.7x `pool_count`'s (~0.9) and drops to a much more GLM-friendly ~0.6 once log-transformed — confirming the concern flagged in the task spec. `pool_count` itself is mildly right-skewed but nowhere near as extreme, and it's already working fine in the existing §3.3 model (IRR, CIs, and diagnostics all sensible), so per the "don't change what isn't broken" principle it is left untransformed. `population_density_per_km2` is used in **log form** (`log_population_density`) throughout this section — confirmed necessary below, not just a precaution: the raw-scale density model fails to converge.

In [ ]:
REDUCED_PLUS = REDUCED_PREDICTOR_COLS  # sgb_ii_recipients_pct, foreign_population_pct, unemployment_total,
                                        # turnout_pct, second_vote_spd_pct, second_vote_bsw_pct

def fit_reduced(cols):
    Xv = sm.add_constant(analysis_df[cols])
    return Xv, NegativeBinomial(y, Xv, offset=offset).fit(disp=False)

# Variant 1: pool_count-based -- this is exactly nb_full from §3.3, refit here under a
# variant-specific name for a clean side-by-side comparison.
X_v_pool, nb_v_pool = fit_reduced(["pool_count"] + REDUCED_PLUS)

# Variant 1b: does the raw (untransformed) density scale even converge? (diagnostic only)
X_v_density_raw, nb_v_density_raw = fit_reduced(["population_density_per_km2"] + REDUCED_PLUS)
print(f"Raw-scale density model converged: {nb_v_density_raw.mle_retvals.get('converged')}")

# Variant 2: density-only (log scale)
X_v_density, nb_v_density = fit_reduced(["log_population_density"] + REDUCED_PLUS)

# Variant 3: both together (log density + pool_count)
X_v_both, nb_v_both = fit_reduced(["pool_count", "log_population_density"] + REDUCED_PLUS)

vif_both = pd.DataFrame({
    "predictor": X_v_both.columns,
    "vif": [variance_inflation_factor(X_v_both.values, i) for i in range(X_v_both.shape[1])],
}).drop(index=0)
vif_both.sort_values("vif", ascending=False)

**Convergence and VIF.** The raw-scale density model fails to converge (`converged: False`) -- confirming the skew concern isn't just cosmetic, it breaks the MLE outright, exactly as the log-transform diagnostic above predicted. The log-scale density model converges cleanly.

For the "both together" spec, `log_population_density`'s VIF is **7.24** against `pool_count`'s 2.44 -- elevated, and by far the third-highest VIF in that model after the already-known `sgb_ii_recipients_pct`/`unemployment_total` pair (~26-29 each, unchanged from §3.2), but it stays under the conventional VIF>10 danger threshold. So the two aren't so collinear that a joint model is uninterpretable the way the 19-covariate model in §3.2 was -- but as the LR test below shows, "not fatally collinear" isn't the same as worth including both.

In [ ]:
def cv_mae_variant(cols, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    Xcv = sm.add_constant(analysis_df[cols])
    rate_actual = analysis_df["racialised_rate"].values
    errors = []
    for train_idx, test_idx in kf.split(np.arange(len(analysis_df))):
        model = NegativeBinomial(y.iloc[train_idx], Xcv.iloc[train_idx], offset=offset.iloc[train_idx]).fit(disp=False)
        pred_rate = np.exp(Xcv.iloc[test_idx].values @ model.params.drop("alpha").values)
        errors.append(np.abs(pred_rate - rate_actual[test_idx]))
    return np.concatenate(errors)

variant_cols = {
    "pool_count-based":  ["pool_count"] + REDUCED_PLUS,
    "density-only (log)": ["log_population_density"] + REDUCED_PLUS,
    "both together":      ["pool_count", "log_population_density"] + REDUCED_PLUS,
}
variant_models = {"pool_count-based": nb_v_pool, "density-only (log)": nb_v_density, "both together": nb_v_both}

# LR tests: "both" nests both single-covariate variants (drop one term each), so both are
# directly testable; pool_count-only vs density-only are non-nested (can't LR test directly).
lr_add_density = 2 * (nb_v_both.llf - nb_v_pool.llf)
p_add_density = stats.chi2.sf(lr_add_density, 1)
lr_add_pool = 2 * (nb_v_both.llf - nb_v_density.llf)
p_add_pool = stats.chi2.sf(lr_add_pool, 1)

rows = []
for name, cols in variant_cols.items():
    m = variant_models[name]
    mae = cv_mae_variant(cols).mean() * 100
    fp_row = {
        "spec": name, "AIC": m.aic, "BIC": m.bic, "llf": m.llf,
        "cv_mae_pp": mae,
        "foreign_population_pct_IRR": np.exp(m.params["foreign_population_pct"]),
        "foreign_population_pct_p": m.pvalues["foreign_population_pct"],
    }
    rows.append(fp_row)
comparison = pd.DataFrame(rows).set_index("spec")
print(comparison.round(4))

print(f"\nLR test, adding log_population_density to pool_count-based model: "
      f"LR={lr_add_density:.3f}, df=1, p={p_add_density:.4f}")
print(f"LR test, adding pool_count to density-only model: "
      f"LR={lr_add_pool:.3f}, df=1, p={p_add_pool:.4f}")

**Takeaway -- three-way comparison and recommendation.**

| Spec | AIC | BIC | 5-fold CV MAE (pp) | `foreign_population_pct` IRR (p) |
|---|---|---|---|---|
| `pool_count`-based (current, §3.3) | 2200.6 | 2233.9 | 0.610 | 1.034 (p<0.0001) |
| Density-only (log) | 2202.8 | 2236.1 | 0.613 | 1.026 (p=0.014) |
| Both together | 2201.6 | 2238.6 | 0.615 | 1.027 (p=0.010) |

Three findings:

1. **`foreign_population_pct` is robust across every specification tried.** IRR ranges narrowly from 1.026 to 1.034 (a 2.6-3.4pp effect per 1pp of foreign-population share), and it is significant at p<0.05 in all three, p<0.0001 in the primary spec. Swapping the density covariate does not touch this finding -- exactly what notebook 02's independent confirmation would predict.
2. **The `pool_count`-based model still edges out both alternatives on every metric** (lowest AIC/BIC, lowest CV MAE), though the margins are small (AIC differs by ~1-2 points, CV MAE by ~0.005pp) -- these are not decisive wins, more like a photo finish. Consistent with that, the LR tests confirm neither addition earns its extra parameter: adding `log_population_density` to the `pool_count`-based model is not significant (LR=0.98, p=0.32), and adding `pool_count` to the density-only model is only marginal (LR=3.20, p=0.074).
3. **We do not adopt the "both together" specification.** Even though its VIF (7.24 for `log_population_density`) stays just under the conventional danger threshold and the model is technically estimable, the LR tests above show it isn't buying real explanatory power over either single-covariate variant -- it's just spending a degree of freedom on two measures of essentially the same underlying construct (pool density vs population density, ρ=-0.80).

**Recommendation: keep `pool_count` as the primary density covariate** (`MODEL_COLS` and `nb_full` in §3.2-3.3 are unchanged). It has a marginal but consistent edge in fit and out-of-sample prediction, and -- unlike `population_density_per_km2` -- it doesn't need a log-transform to be usable, which keeps the existing IRR-based interpretation simple. This is a closer call than notebook 02's raw correlation numbers alone would suggest (density "wins" ρ=0.54 vs -0.49 there), and it's worth being honest about why: once `foreign_population_pct` and the other five predictors are already in the model, whichever density measure is added contributes comparatively little either way, and `pool_count`'s advantage is small enough that a future iteration with a cleaner or updated density measure could plausibly flip this. For now, `population_density_per_km2` (log-transformed) is flagged as a close, theoretically well-motivated second choice worth revisiting rather than a decisively rejected alternative -- and the log-transform is a **permanent requirement if it's ever adopted**, not just a one-off diagnostic check for this comparison (the raw scale genuinely fails to converge).

## 3.5 Robustness check: restricting to the `reliable_n` constituencies

Same check §2.4 ran for the univariate correlations -- refit the model restricted to constituencies clearing §1.2's reliability bar. **Note:** `descriptive_analysis.ipynb` tightened this bar from `social_reviews>=100 AND racialised_reviews>=5` to `social_reviews>=100 AND racialised_reviews>=10` -- so the reliable subset is now smaller than it was when this notebook was first built (247/299 pass, not the 288/299 the bar used to yield; that number is computed directly from `analysis_df["reliable_n"]` just below rather than hardcoded, so it always reflects whatever the upstream notebook currently produces). We compare coefficients to the all-299 model to see whether any result is being driven by the (now 52, not 11) thinner constituencies excluded by the stricter bar.

In [ ]:
n_reliable = int(analysis_df["reliable_n"].sum())
print(f"reliable_n constituencies: {n_reliable} / {len(analysis_df)}")

reliable_df = analysis_df[analysis_df["reliable_n"]]
X_rel = sm.add_constant(reliable_df[MODEL_COLS])
y_rel = reliable_df["racialised_reviews"]
offset_rel = reliable_df["log_social_reviews"]

nb_reliable = NegativeBinomial(y_rel, X_rel, offset=offset_rel).fit(disp=False)

coef_compare = pd.DataFrame({
    "coef_all_299": nb_full.params,
    f"coef_reliable_{n_reliable}": nb_reliable.params,
}).drop(index="alpha", errors="ignore")
coef_compare["shift"] = coef_compare[f"coef_reliable_{n_reliable}"] - coef_compare["coef_all_299"]
coef_compare.sort_values("shift", key=np.abs, ascending=False)

**Takeaway.** All 7 coefficients are stable between the 299-constituency and the (now 247, corrected from a stale 288 -- see §3.5 header) `reliable_n` fits -- the largest shifts are `unemployment_total` (-0.039) and `sgb_ii_recipients_pct` (+0.026), both larger in absolute terms than the old 288-subset check reported (expected, since the stricter `>=10` bar now excludes 52 constituencies rather than 11, a much bigger and more different subset), but still small relative to their standard errors in §3.3 and every coefficient keeps its sign. None of §3.3's findings are being driven by the thin constituencies the reliability bar screens out, matching the pattern §2.4's correlation robustness check already found -- the stricter threshold changes *which* constituencies are excluded but not the substantive conclusion.

## 3.6 Do model residuals still show spatial clustering?

§2.6 found genuine spatial clustering in the raw `racialised_rate` (Moran's I = 0.28). If the covariates in this model fully explained that clustering, the model's residuals should look spatially random; if Moran's I on the residuals is still significant, that's a sign the NB2 specification is missing something structurally spatial (e.g. a spatial lag or a state-level random effect) — a concrete, flagged item for the next iteration, not something to paper over by adding more predictors to this pass.

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran

mu_full = np.exp(X.values @ nb_full.params.drop("alpha").values + offset.values)
alpha_hat = nb_full.params["alpha"]
var_full = mu_full + alpha_hat * mu_full ** 2
analysis_df["pearson_resid"] = (y.values - mu_full) / np.sqrt(var_full)

constituencies = gpd.read_file(SHAPEFILE_PATH)
resid_map_df = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    analysis_df[["constituency_code", "pearson_resid"]], on="constituency_code", how="left"
).dropna(subset=["pearson_resid"]).reset_index(drop=True)

w = Queen.from_dataframe(resid_map_df, use_index=False)
w.transform = "r"
moran_resid = Moran(resid_map_df["pearson_resid"].values, w, permutations=999)

print(f"Moran's I on NB2 Pearson residuals = {moran_resid.I:.4f}")
print(f"p-value (999 permutations) = {moran_resid.p_sim:.4f}")
print(f"z-score = {moran_resid.z_sim:.2f}")
print(f"\nFor comparison, Moran's I on the raw racialised_rate (\u00a72.6) = 0.28, p = 0.001")

In [ ]:
# Task 1 settled on keeping pool_count as the primary specification (§3.4b), so the
# check above (on nb_full's residuals) remains the primary result. For completeness,
# we also check whether the density-only (log) variant's residuals show materially
# different spatial clustering -- i.e. whether the choice of density covariate matters
# for the unexplained spatial structure, not just for in-sample/CV fit.
mu_density = np.exp(X_v_density.values @ nb_v_density.params.drop("alpha").values + offset.values)
alpha_density = nb_v_density.params["alpha"]
var_density = mu_density + alpha_density * mu_density ** 2
resid_density = (y.values - mu_density) / np.sqrt(var_density)

resid_map_density = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    analysis_df[["constituency_code"]].assign(pearson_resid=resid_density), on="constituency_code", how="left"
).dropna(subset=["pearson_resid"]).reset_index(drop=True)

moran_density = Moran(resid_map_density["pearson_resid"].values, w, permutations=999)
print(f"Moran's I, pool_count-based model residuals (primary, above)  = {moran_resid.I:.4f} "
      f"(p={moran_resid.p_sim:.3f})")
print(f"Moran's I, density-only (log) model residuals                 = {moran_density.I:.4f} "
      f"(p={moran_density.p_sim:.3f})")

**Takeaway (density-covariate comparison).** The density-only variant's residual Moran's I (~0.11) is essentially the same as the pool_count-based model's (0.118) -- the two differ by less than either model's own permutation noise would allow us to distinguish. Swapping which density covariate goes into the model does not meaningfully change the amount of residual spatial clustering left over; the unexplained spatial structure is a property of the outcome and the predictor set as a whole (most likely something like a state-level effect or genuine spatial autocorrelation in unmeasured factors), not an artifact of choosing `pool_count` over `population_density_per_km2`. This reinforces §3.6's original conclusion that a spatial-lag/error term or state random effect is the right next step, independent of which density measure the model eventually settles on.

**Takeaway.** Moran's I on the model's Pearson residuals is 0.118 (p=0.004, z=3.50) — still significant, but less than half of the raw `racialised_rate`'s I=0.28. So this model explains part of the spatial clustering §2.6 found (mainly through `foreign_population_pct`, which is itself spatially structured — the Ruhr and Berlin hotspots are both higher-foreign-population-share areas), but a meaningful amount of spatial structure remains unexplained. This is the clearest concrete item for the next iteration: a spatial-lag/spatial-error NB2, or a state-level random effect (cheaper to implement), rather than adding more socioeconomic covariates to this specification.

## 3.7 Part 3 synthesis: what this preliminary model shows

**What this preliminary model supports:**
- A genuine, now triply-confirmed effect of `foreign_population_pct` on the racialised-review rate, net of venue density and five other predictors (IRR=1.034, p<0.0001 in the primary `pool_count`-based spec) -- the single most robust quantitative finding across all three notebooks. §3.4b confirms this holds up (IRR 1.026-1.034, p<0.05 in every case) whether the density covariate is `pool_count`, `population_density_per_km2` (log), or both together -- it is not an artifact of which density measure is chosen.
- `pool_count` remains the recommended density covariate, but only marginally. §3.4b tested `population_density_per_km2` as an alternative, motivated by `02_socioeconomic_political_relations.ipynb`'s finding that density is the more fundamental driver in raw correlation terms (density vs `racialised_rate` ρ=0.597 vs `pool_count`'s ρ=-0.556, with the two strongly collinear at ρ=-0.794). Once folded into this notebook's full multivariate NB2 alongside `foreign_population_pct` and the other five predictors, that raw-correlation advantage does not carry over: `pool_count` still edges out density-only and both-together on AIC (2200.6 vs 2202.8 vs 2201.6), BIC, and 5-fold CV MAE (0.610pp vs 0.613pp vs 0.615pp), and LR tests confirm adding the other density measure to either single-covariate model does not earn its extra parameter (p=0.32 and p=0.074 respectively). The margins are small, so this is a marginal, revisable call, not a decisive rejection of density.
- The 6-predictor set as a whole adds real explanatory power over `pool_count` alone (LR=77.2, p=1.3e-14) and improves out-of-sample prediction (5-fold CV MAE 0.61pp vs. 0.69pp) -- a modest but genuine predictive gain.
- Coefficients are stable under the `reliable_n` robustness check (§3.5) -- now correctly **247/299** constituencies (corrected from a stale 288/299 in this section, which predated `descriptive_analysis.ipynb`'s tightening of the reliability bar from `racialised_reviews>=5` to `>=10`). All 7 coefficients keep their sign on the stricter 247-constituency subset; the largest shifts (`unemployment_total` -0.039, `sgb_ii_recipients_pct` +0.026) are larger than the old check reported, consistent with a bigger, more different set of excluded constituencies (52 now vs. 11 before), but still well within the range expected from sampling noise given the coefficients' standard errors in §3.3.

**What this preliminary model does not (yet) support, and why:**
- Individual effects of `sgb_ii_recipients_pct` and `unemployment_total` -- both dip below significance once modelled jointly (VIF 26.8/25.8, §3.2), because they're too collinear with each other to separate. §2.4b's partial correlations should be read as evidence of a shared "deprivation" signal here, not as two independent effects. (Notebook 02 also flags that 20 constituencies -- Berlin 12, Hamburg 6, Bremen 2 -- share state-wide-constant values on several of the ingredient columns behind these two predictors and others; this is a data-granularity caveat rather than a bug, and it applies to this model too. It did not warrant re-running the full robustness suite excluding those 20 here, since notebook 02 found no sign flips and only modest (up to 0.09) shifts when doing so at the correlation level.)
- `turnout_pct`, `second_vote_spd_pct`, `second_vote_bsw_pct` -- all non-significant once placed alongside the stronger predictors; their §2.4b partial correlations were real but too small to survive joint estimation at n=299.
- A model that fully explains the spatial clustering §2.6 found: residual Moran's I=0.12 (p=0.004) is roughly half the raw rate's I=0.28, meaning genuine spatial structure remains after controlling for these six predictors and venue density. §3.6's extended check confirms this is not sensitive to the `pool_count`-vs-density choice either (density-only variant's residual I~0.11 is statistically indistinguishable from the primary model's).
- A model using all 18 §2.4 predictors together, tried and explicitly rejected in §3.2 for this pass (VIF up to 79.7, MLE fails to converge) -- a full accounting of every predictor isn't viable at n=299 without either dimensionality reduction (e.g. a deprivation composite index) or a different estimation approach (e.g. regularisation).
- A joint `pool_count` + `population_density_per_km2` specification (§3.4b): technically estimable (VIF 7.24, under the conventional danger threshold) but not supported by the LR tests, so not adopted. If `population_density_per_km2` is used in a future iteration, note that **the log-transform is a permanent requirement, not just a diagnostic check for this round** -- the raw scale is skewed enough (skew~3.3 vs. `pool_count`'s ~0.9) that the untransformed NB2 fails to converge outright.

**Concrete next-iteration items, in priority order:** (1) a spatial-lag/error NB2 or state-level random effect to address the residual clustering in §3.6, which §3.4b confirms is present regardless of the density covariate used; (2) resolve the `sgb_ii_recipients_pct`/`unemployment_total` collinearity, either by combining them into one deprivation index or picking one on theoretical grounds; (3) revisit whether `population_density_per_km2` should replace `pool_count` once a further iteration either resolves the density/pool_count near-tie more decisively or brings in additional predictors that might interact differently with a true density measure than with a venue count; (4) revisit once `theme`/`approach` IAA is finalised, since the outcome label itself (`racialised`) is what all of this is built on. This is a first, deliberately simple predictive pass, consistent with the plan agreed before starting -- the model, predictor set, and diagnostics above are all expected to be revised as that work continues.